In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# Load CIFAR-10 dataset
df = pd.read_csv('cifar-10.csv').dropna()

# Separate images and labels
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values.astype(int)

# Reshape images into 32x32 RGB format and normalize
X = X.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
X = X / 255.0

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Build CNN model
model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),

    # First convolution block
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Second convolution block
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Third convolution block
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Classify the extracted features
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Train the model (10% of training data is used for validation)
history = model.fit(
    X_train, y_train,
    epochs=8,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

# Evaluate on unseen test images
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print("Test Accuracy:", test_acc * 100, "%")
print("Test Loss:", test_loss)

# Predict the test images
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

# Display confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("CIFAR-10 Confusion Matrix")
plt.show()


# Display 10 test images with actual and predicted labels
plt.figure(figsize=(12, 6))

for i in range(10):
    plt.subplot(2, 5, i + 1)

    # Display the image
    plt.imshow(X_test[i])

    # Show actual and predicted class names
    actual = class_names[y_test[i]]
    predicted = class_names[y_pred[i]]

    plt.title(
        "Actual: " + actual +
        "\nPredicted: " + predicted,
        color='green' if y_test[i] == y_pred[i] else 'red',
        fontsize=9
    )

    plt.axis('off')

plt.tight_layout()
plt.show()

